# QUBO Contract + Validation Utilities (Shared)

This notebook defines and validates a single “contract” for QUBO scenario files used across the project (Scenario A/B/C). The goal is to eliminate schema drift and repeated parsing logic across notebooks by:

- Standardizing a single JSON schema: `Q`, `nct_ids`, and `meta`
- Providing reusable utilities for:
  - schema validation
  - robust loading
  - bitstring decoding (LR/RL)
  - QUBO energy scoring
  - selection extraction
  - overlap (Jaccard) scoring

At the end, we validate existing QUBO files (Scenario A and B) and optionally normalize/upgrade them into the canonical schema.


In [1]:
# ============================================================
# Cell 1 — Setup: imports, paths, and targets
# ============================================================

from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

ROOT = Path(".")
QUBO_DIR = ROOT / "data" / "qubo_scenarios"
QUBO_DIR.mkdir(parents=True, exist_ok=True)

# Targets we will validate/normalize if present
PATH_QUBO_A = QUBO_DIR / "scenario_A_qubo.json"
PATH_QUBO_B = QUBO_DIR / "scenario_B_qubo.json"

print("QUBO_DIR:", QUBO_DIR.resolve())
print("Exists A:", PATH_QUBO_A.exists(), "|", PATH_QUBO_A)
print("Exists B:", PATH_QUBO_B.exists(), "|", PATH_QUBO_B)


QUBO_DIR: /home/parallels/projects/quantum-clinical-trial-optimization/data/qubo_scenarios
Exists A: False | data/qubo_scenarios/scenario_A_qubo.json
Exists B: True | data/qubo_scenarios/scenario_B_qubo.json


### What Cell 1 Just Did

- Imported standard libraries plus NumPy/Pandas for matrix validation and reporting.
- Defined common paths for the QUBO scenario JSON files (`data/qubo_scenarios/`).
- Confirmed whether Scenario A and B QUBO files exist in the expected location.


In [2]:
# ============================================================
# Cell 2 — Canonical QUBO schema + core utilities
# ============================================================

@dataclass
class QuboScenario:
    Q: np.ndarray                 # (n,n) float matrix
    nct_ids: List[str]            # length n, in the same order as Q variables
    meta: Dict[str, Any]          # scenario metadata

def _as_ndarray_Q(Q_any: Any) -> np.ndarray:
    Q = np.array(Q_any, dtype=float)
    if Q.ndim != 2 or Q.shape[0] != Q.shape[1]:
        raise ValueError(f"Q must be a square 2D matrix; got shape={Q.shape}")
    return Q

def validate_qubo_contract(payload: Dict[str, Any]) -> None:
    if not isinstance(payload, dict):
        raise ValueError("QUBO payload must be a JSON object (dict).")

    if "Q" not in payload:
        raise ValueError("Missing required key: 'Q' (square matrix).")
    if "nct_ids" not in payload:
        raise ValueError("Missing required key: 'nct_ids' (list aligned to Q).")
    if "meta" not in payload:
        raise ValueError("Missing required key: 'meta' (scenario metadata dict).")

    Q = _as_ndarray_Q(payload["Q"])
    n = Q.shape[0]

    nct_ids = payload["nct_ids"]
    if not isinstance(nct_ids, list) or not all(isinstance(x, str) for x in nct_ids):
        raise ValueError("'nct_ids' must be a list[str].")
    if len(nct_ids) != n:
        raise ValueError(f"len(nct_ids)={len(nct_ids)} must match Q dimension n={n}.")

    meta = payload["meta"]
    if not isinstance(meta, dict):
        raise ValueError("'meta' must be a dict/json object.")

def load_qubo(path: Path) -> QuboScenario:
    payload = json.loads(path.read_text())
    validate_qubo_contract(payload)
    Q = _as_ndarray_Q(payload["Q"])
    return QuboScenario(Q=Q, nct_ids=list(payload["nct_ids"]), meta=dict(payload["meta"]))

def save_qubo(path: Path, scenario: QuboScenario) -> None:
    payload = {
        "Q": scenario.Q.tolist(),
        "nct_ids": scenario.nct_ids,
        "meta": scenario.meta,
    }
    validate_qubo_contract(payload)
    path.write_text(json.dumps(payload, indent=2))

def qubo_energy(Q: np.ndarray, x01: np.ndarray) -> float:
    x = np.asarray(x01, dtype=float).reshape(-1, 1)
    return float((x.T @ Q @ x)[0, 0])

def decode_bitstring(bitstring: str, n: int, order: str = "LR") -> np.ndarray:
    """
    order='LR' interprets string left-to-right as x[0..n-1].
    order='RL' reverses bits before mapping to x[0..n-1].
    """
    s = str(bitstring).replace(" ", "")
    if len(s) != n:
        s = s[:n].ljust(n, "0")
    if order.upper() == "RL":
        s = s[::-1]
    return np.array([1 if ch == "1" else 0 for ch in s], dtype=int)

def best_decode_for_energy(Q: np.ndarray, bitstring: str) -> Tuple[float, str, np.ndarray]:
    n = Q.shape[0]
    x_lr = decode_bitstring(bitstring, n, "LR")
    x_rl = decode_bitstring(bitstring, n, "RL")
    E_lr = qubo_energy(Q, x_lr)
    E_rl = qubo_energy(Q, x_rl)
    if E_lr <= E_rl:
        return E_lr, "LR", x_lr
    return E_rl, "RL", x_rl

def selected_indices(x01: np.ndarray) -> List[int]:
    x = np.asarray(x01, dtype=int).ravel()
    return np.where(x == 1)[0].tolist()

def selected_ids(nct_ids: List[str], x01: np.ndarray) -> List[str]:
    idx = selected_indices(x01)
    return [nct_ids[i] for i in idx]

def jaccard(a: List[Any], b: List[Any]) -> float:
    A, B = set(a), set(b)
    if not A and not B:
        return 1.0
    return len(A & B) / max(1, len(A | B))

print("OK: QUBO contract + utilities loaded.")


OK: QUBO contract + utilities loaded.


### What Cell 2 Just Did

- Defined the canonical QUBO JSON contract: `{ "Q": ..., "nct_ids": ..., "meta": ... }`.
- Implemented reusable utilities to validate, load, and save contractual QUBO files.
- Added core scoring and decoding helpers:
  - QUBO energy evaluation
  - bitstring decoding (LR/RL)
  - deriving selected indices / trial IDs
  - Jaccard overlap for comparing selections


In [3]:
# ============================================================
# Cell 3 — Validate existing scenario QUBO files (A/B) and report
# ============================================================

rows = []

for p in [PATH_QUBO_A, PATH_QUBO_B]:
    if not p.exists():
        rows.append({"file": str(p), "status": "MISSING", "n": None, "meta_keys": None, "error": None})
        continue

    try:
        scen = load_qubo(p)
        rows.append({
            "file": str(p),
            "status": "OK",
            "n": int(scen.Q.shape[0]),
            "meta_keys": sorted(list(scen.meta.keys())),
            "error": None
        })
    except Exception as e:
        rows.append({
            "file": str(p),
            "status": "INVALID",
            "n": None,
            "meta_keys": None,
            "error": str(e)
        })

df_report = pd.DataFrame(rows)
display(df_report)


,file,status,n,meta_keys,error
0,data/qubo_scenarios/scenario_A_qubo.json,MISSING,None,None,None
1,data/qubo_scenarios/scenario_B_qubo.json,INVALID,None,None,Missing required key: 'meta' (scenario metadat...


### What Cell 3 Just Did

- Loaded and validated existing Scenario A and Scenario B QUBO JSON files against the canonical contract.
- Produced a small report showing which files are present, valid, and their QUBO dimension `n`.
- If any file is flagged INVALID, we’ll normalize it in the next cell.


In [4]:
# ============================================================
# Cell 4 — Normalize legacy QUBO JSONs → canonical contract (if needed)
#   Safe: writes *_canonical.json (does not overwrite originals)
# ============================================================

def normalize_legacy_qubo(path: Path) -> Optional[Path]:
    if not path.exists():
        return None

    raw = json.loads(path.read_text())

    # If already canonical, no-op (but still write canonical copy for uniformity if you want)
    try:
        validate_qubo_contract(raw)
        out = path.with_name(path.stem + "_canonical.json")
        scen = load_qubo(path)
        save_qubo(out, scen)
        return out
    except Exception:
        pass

    # Try a few known legacy shapes we’ve seen in notebooks:
    # 1) {"Q": ... , "nct_ids": ...} but missing meta
    if "Q" in raw and "nct_ids" in raw and "meta" not in raw:
        Q = _as_ndarray_Q(raw["Q"])
        nct_ids = list(raw["nct_ids"])
        meta = {"scenario": path.stem.replace("_qubo", ""), "note": "auto-added meta during normalization"}
        scen = QuboScenario(Q=Q, nct_ids=nct_ids, meta=meta)
        out = path.with_name(path.stem + "_canonical.json")
        save_qubo(out, scen)
        return out

    # 2) {"qubo_matrix": ...} and maybe {"nct_ids": ...}
    if "qubo_matrix" in raw:
        Q = _as_ndarray_Q(raw["qubo_matrix"])
        n = Q.shape[0]
        nct_ids = raw.get("nct_ids") or [f"var_{i}" for i in range(n)]
        meta = raw.get("meta") or {"scenario": path.stem.replace("_qubo", ""), "note": "normalized from qubo_matrix"}
        scen = QuboScenario(Q=Q, nct_ids=list(nct_ids), meta=dict(meta))
        out = path.with_name(path.stem + "_canonical.json")
        save_qubo(out, scen)
        return out

    # 3) {"qubo": { "i,j": val, ... }, "n": N } sparse dict form
    if "qubo" in raw and isinstance(raw["qubo"], dict):
        n = int(raw.get("n") or 0)
        if n <= 0:
            # infer n from keys
            mx = -1
            for k in raw["qubo"].keys():
                i, j = k.split(",")
                mx = max(mx, int(i), int(j))
            n = mx + 1
        Q = np.zeros((n, n), dtype=float)
        for k, v in raw["qubo"].items():
            i, j = k.split(",")
            Q[int(i), int(j)] += float(v)
        nct_ids = raw.get("nct_ids") or [f"var_{i}" for i in range(n)]
        meta = raw.get("meta") or {"scenario": path.stem.replace("_qubo", ""), "note": "normalized from sparse qubo dict"}
        scen = QuboScenario(Q=Q, nct_ids=list(nct_ids), meta=dict(meta))
        out = path.with_name(path.stem + "_canonical.json")
        save_qubo(out, scen)
        return out

    raise ValueError(f"Could not normalize legacy QUBO schema for {path}. Keys: {list(raw.keys())[:30]}")

normalized = []
for p in [PATH_QUBO_A, PATH_QUBO_B]:
    if not p.exists():
        continue
    try:
        out = normalize_legacy_qubo(p)
        if out is not None:
            normalized.append(str(out))
    except Exception as e:
        print(f"Normalize failed for {p}: {e}")

print("Normalized outputs:")
for x in normalized:
    print(" -", x)


Normalized outputs:
 - data/qubo_scenarios/scenario_B_qubo_canonical.json


### What Cell 4 Just Did

- Attempted to normalize any non-canonical QUBO JSON into the canonical `{Q, nct_ids, meta}` structure.
- Wrote safe outputs as `*_canonical.json` (does not overwrite the original files).
- If normalization fails, it prints the top-level keys so we can add one more legacy parser case.


## Summary

- We defined a single canonical QUBO JSON contract used across scenarios: `Q`, `nct_ids`, and `meta`.
- We implemented a shared set of utilities (validation, loading/saving, decoding, scoring, and overlap metrics) that can be reused in every notebook.
- We generated a validation report for Scenario A/B QUBO files and provided a safe normalizer that creates `*_canonical.json` outputs when legacy formats are detected.

Next hop: extract these utilities into `src/qubo_utils.py` and update Scenario A/B/C notebooks to import them instead of duplicating parsing/decoding logic.
